# 第 1 章：KV Cache

本 Notebook 用一个小 Attention 演示 KV Cache。目标不是生成自然语言，而是观察：首次 Prefill 后，Cache 长度如何随每个新 token 增长。免费 CPU 足够。

## 1. 先算 KV Cache 需要多少显存

`2 × 层数 × KV头数 × 每头维度 × token数 × batch × 每元素字节数`。

下面用 Llama 3 8B 的近似配置：32 层、8 个 KV 头、每头 128 维、FP16（每个数字 2 字节）。

In [ ]:
def kv_cache_bytes(layers, kv_heads, head_dim, seq_len, batch=1, bytes_per_element=2):
    return 2 * layers * kv_heads * head_dim * seq_len * batch * bytes_per_element

size = kv_cache_bytes(layers=32, kv_heads=8, head_dim=128, seq_len=8192)
print(f'8K token 的 KV Cache: {size / 1024**3:.2f} GiB')
print(f'每个 token: {size / 8192 / 1024:.0f} KiB')

## 2. 一个支持 Cache 的 Attention

第一次运行时 `cache=None`，模型计算 prompt 的 K 和 V。之后每一步只传入一个新 token；旧 K、V 从 `cache` 取出，新 K、V 追加到末尾。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class CachedAttention(nn.Module):
    def __init__(self, d=32, n_head=4):
        super().__init__()
        assert d % n_head == 0
        self.n_head, self.head_dim = n_head, d // n_head
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)

    def forward(self, x, cache=None):
        B, S, D = x.shape
        q, k, v = self.qkv(x).view(B, S, 3, self.n_head, self.head_dim).unbind(2)
        q, k, v = [z.transpose(1, 2) for z in (q, k, v)]

        if cache is not None:
            old_k, old_v = cache
            k = torch.cat([old_k, k], dim=2)  # 玩具写法；生产系统用分页块避免复制
            v = torch.cat([old_v, v], dim=2)
        new_cache = (k, v)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if cache is None:  # Prefill 时需要遮住未来 token
            mask = torch.tril(torch.ones(S, S, dtype=torch.bool))
            scores = scores.masked_fill(~mask, float('-inf'))
        y = F.softmax(scores, dim=-1) @ v
        y = y.transpose(1, 2).contiguous().view(B, S, D)
        return self.proj(y), new_cache

## 3. Prefill 一次，再 Decode 多次

先喂 4 个 token，得到长度为 4 的 Cache。然后每次只喂 1 个 token。输出中的 `KV 长度` 应依次为 `4 → 5 → 6 → 7`。

In [ ]:
attn = CachedAttention()

# Prefill：一次处理整段 prompt
prompt = torch.randn(1, 4, 32)
_, cache = attn(prompt)
print('Prefill 后 KV 长度:', cache[0].shape[2])

# Decode：每次仅处理 1 个新 token
for step in range(3):
    new_token = torch.randn(1, 1, 32)
    _, cache = attn(new_token, cache)
    print(f'Decode 第 {step + 1} 步后 KV 长度:', cache[0].shape[2])

## 小练习

1. 把 prompt 长度从 4 改成 64，观察第一次 Cache 长度。
2. 把 `n_head=4` 改成 8，查看 Cache 的 head 维度。
3. 注意 `torch.cat` 会复制旧数据。这是教学用写法；vLLM 的 PagedAttention 在后续章节会解决它。